<a href="https://colab.research.google.com/github/manuelagutierrezss16/Integraci-n-de-Datos-y-Prospectiva/blob/main/4_1_Integraci%C3%B3n_Multidimencsional_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CASO DE ESTUDIO**

Una entidad prestadora de servicios de salud (EPS) del sistema de
atención en salud en Colombia, requiere mejorar la eficiencia de sus
operaciones del negocio, por lo cual debe reducir su tamaño. (optimización de operaciones integrando diferentes sucursales ubicadas en el Valle de Aburra.)
En este sentido, la EPS quiere centrar sus operaciones en los sectores más cercanos a la ciudad de Medellín por lo que quiere hacer la integración de los pacientes de Coopacabana y Caldas en sus otras sedes. Dentro de los objetivos, quiere evaluar como era la configuración de sus variables para antes y despues de la integracíon de pacientes nuevos.

En este momento vamos a identificar a donde enviar a los pacientes de Caldas y Copacabana teniendo en cienta las siguientes variables:
* Colocar la descripcion de cada una de las variables de trabajo.

0. Cargar librerias


In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Se cargan los datos de trabajo

In [27]:
nxl='/content/drive/MyDrive/Colab Notebooks/INTEGRACION/BASES DE DDATOS/5. Diabetes Árbol_Int_Mult.xlsx'
XDB=pd.read_excel (nxl,sheet_name=0)
XDB=XDB.dropna()
XDB.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Numero_Atenciones,Costo_Promedio_Atencion,Ciudad_Pertenencia
0,0,125,96,0,0,22.5,0.262,21,0,4,314.926044,Sabaneta
1,0,141,0,0,0,42.4,0.205,29,1,6,287.355812,Envigado
2,10,101,86,37,0,45.6,1.136,38,1,8,464.541158,Sabaneta
3,1,96,122,0,0,22.4,0.207,27,0,3,368.103292,Sabaneta
4,5,139,64,35,140,28.6,0.411,26,0,9,428.548590,Copacabana


In [28]:
#SE CONSTRUYE LA SEGUNDA BASE DE DATOS (XDB2) PARA CLASIFICAR POR DIABETES Y CIUDAD (la misma base de datos pero lo que necesitamos al final)
XDB2=XDB.iloc[:,[0,1,2,3,4,5,6,7,9,10,8,11]].copy()
display(XDB2)

XDB=XDB.iloc[:,[0,1,2,3,4,5,6,7,9,10]] # Solo están las variables de entrada
display(XDB)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion,Outcome,Ciudad_Pertenencia
0,0,125,96,0,0,22.5,0.262,21,4,314.926044,0,Sabaneta
1,0,141,0,0,0,42.4,0.205,29,6,287.355812,1,Envigado
2,10,101,86,37,0,45.6,1.136,38,8,464.541158,1,Sabaneta
3,1,96,122,0,0,22.4,0.207,27,3,368.103292,0,Sabaneta
4,5,139,64,35,140,28.6,0.411,26,9,428.548590,0,Copacabana
...,...,...,...,...,...,...,...,...,...,...,...,...
2495,1,118,58,36,94,33.3,0.261,23,3,343.603552,0,Caldas
2496,1,92,62,25,41,19.5,0.482,25,3,94.845893,0,Copacabana
2497,4,123,80,15,176,32.0,0.443,34,7,194.316790,0,Copacabana
2498,5,123,74,40,77,34.1,0.269,28,9,126.343793,0,Sabaneta


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion
0,0,125,96,0,0,22.5,0.262,21,4,314.926044
1,0,141,0,0,0,42.4,0.205,29,6,287.355812
2,10,101,86,37,0,45.6,1.136,38,8,464.541158
3,1,96,122,0,0,22.4,0.207,27,3,368.103292
4,5,139,64,35,140,28.6,0.411,26,9,428.548590
...,...,...,...,...,...,...,...,...,...,...
2495,1,118,58,36,94,33.3,0.261,23,3,343.603552
2496,1,92,62,25,41,19.5,0.482,25,3,94.845893
2497,4,123,80,15,176,32.0,0.443,34,7,194.316790
2498,5,123,74,40,77,34.1,0.269,28,9,126.343793


2. Se procede con la cración de los clusters para las EPS de los minicipios que van a quedar activos

In [31]:
Xciu= XDB2['Ciudad_Pertenencia'].unique()
Xciu= ['Sabaneta', 'Envigado', 'Medellín', 'Bello'] # Estos serán los concentradores de información
# Se procede con la creación de los clusters
XCm= np.zeros((len(Xciu),10)) # La variable 8 y 11 no se usan porque son variables de salida


# Fase 0 : reconocer los clusters o los valores de entrada por ciudad
for i, city_name in enumerate(Xciu):
  print(i, city_name)
  filas = np.where(XDB2['Ciudad_Pertenencia']==city_name)[0]# Digame las filas en donde dentro de XDB2 cuales son iguales a city name
  XCm[i,:] = np.mean(XDB.iloc[filas,:],axis=0) # axis 0 promedios de columnas

df= pd.DataFrame(XCm,columns=XDB.columns)
df.index=Xciu
display(df)
#

0 Sabaneta
1 Envigado
2 Medellín
3 Bello


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion
Sabaneta,3.872685,122.824074,67.907407,19.967593,77.511574,31.730093,0.491333,33.488426,5.557870,273.684041
Envigado,3.805699,121.207254,68.829016,20.730570,79.409326,32.046114,0.460345,32.525907,5.515544,282.643785
Medellín,3.820862,120.746032,70.287982,20.977324,82.947846,31.569388,0.489918,33.299320,5.421769,273.701485
Bello,4.123874,122.542793,70.085586,20.614865,78.459459,32.285135,0.481115,33.774775,5.574324,280.357432


3. Se procede con la integración de los pacientes de Caldas y Copacabana

In [37]:
XCiu2=['Caldas','Copacabana']
XCiu3= [] #Aqui voy a almacenar en que EPS quedaron los pacientes
for k in range (len(XDB2)): #Tiene todas las variables completas
  print(XDB2.iloc[k,11]) # A donde pertenece cada paciente

  if XDB2.iloc[k,11]=='Caldas':# or XDB2.iloc[k,11]=='Copacabana': (para borrar todo elif "antes del else")
    d=np.sum((XCm-XDB.iloc[k,:].values)**2,axis=1) # distancia de cada paciente a cada cluster
    nc=np.argmin(d) # dice a cual pertenece la persona
    print(Xciu[nc])
    XCiu3.append(Xciu[nc])

  elif XDB2.iloc[k,11]=='Copacabana':
    d=np.sum((XCm-XDB.iloc[k,:].values)**2,axis=1) # distancia de cada paciente a cada cluster
    nc=np.argmin(d) # dice a cual pertenece la persona
    print(Xciu[nc])
    XCiu3.append(Xciu[nc])

  else:
       XCiu3.append('--')

XDB2['Transferencia']=XCiu3 # Ahora si revisamos XDB2 hay una nueva variable llamda transferencia
display(XDB2)

Sabaneta
Envigado
Sabaneta
Sabaneta
Copacabana
Envigado
Medellín
Medellín
Sabaneta
Bello
Caldas
Envigado
Medellín
Bello
Caldas
Envigado
Envigado
Copacabana
Sabaneta
Envigado
Medellín
Bello
Sabaneta
Sabaneta
Copacabana
Sabaneta
Caldas
Envigado
Caldas
Envigado
Copacabana
Envigado
Sabaneta
Medellín
Copacabana
Sabaneta
Medellín
Sabaneta
Medellín
Bello
Caldas
Envigado
Medellín
Envigado
Copacabana
Sabaneta
Bello
Bello
Envigado
Envigado
Bello
Envigado
Envigado
Bello
Medellín
Medellín
Envigado
Caldas
Envigado
Copacabana
Envigado
Bello
Envigado
Envigado
Copacabana
Envigado
Medellín
Medellín
Bello
Envigado
Medellín
Copacabana
Sabaneta
Medellín
Copacabana
Medellín
Caldas
Bello
Medellín
Copacabana
Medellín
Caldas
Medellín
Envigado
Caldas
Sabaneta
Bello
Caldas
Envigado
Envigado
Copacabana
Envigado
Medellín
Bello
Envigado
Sabaneta
Envigado
Copacabana
Sabaneta
Sabaneta
Medellín
Sabaneta
Bello
Sabaneta
Caldas
Envigado
Caldas
Envigado
Medellín
Envigado
Sabaneta
Medellín
Medellín
Medellín
Caldas
Bello
C

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion,Outcome,Ciudad_Pertenencia,Transferencia
0,0,125,96,0,0,22.5,0.262,21,4,314.926044,0,Sabaneta,--
1,0,141,0,0,0,42.4,0.205,29,6,287.355812,1,Envigado,--
2,10,101,86,37,0,45.6,1.136,38,8,464.541158,1,Sabaneta,--
3,1,96,122,0,0,22.4,0.207,27,3,368.103292,0,Sabaneta,--
4,5,139,64,35,140,28.6,0.411,26,9,428.548590,0,Copacabana,Envigado
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2495,1,118,58,36,94,33.3,0.261,23,3,343.603552,0,Caldas,Envigado
2496,1,92,62,25,41,19.5,0.482,25,3,94.845893,0,Copacabana,Sabaneta
2497,4,123,80,15,176,32.0,0.443,34,7,194.316790,0,Copacabana,Medellín
2498,5,123,74,40,77,34.1,0.269,28,9,126.343793,0,Sabaneta,--
